In [3]:
import pandas as pd
import os, sys, warnings

import earthaccess
from osgeo import gdal

import pandas as pd
import numpy as np
import xarray as xr
import math
import glob

import rasterio as rio

import netCDF4 as nc
from datetime import datetime, timedelta, timezone

from scipy import ndimage as ndi
from scipy.ndimage import binary_fill_holes, center_of_mass, distance_transform_edt
from scipy.interpolate import PchipInterpolator  # monotone, shape-preserving

import matplotlib.pyplot as plt
import matplotlib.patches as patches

from scipy import linalg
from scipy.signal import savgol_filter
from scipy.ndimage import gaussian_filter, binary_erosion, binary_dilation, rotate, shift
from skimage.transform import hough_line, hough_line_peaks
from skimage.restoration import (
    denoise_tv_chambolle,
    denoise_bilateral,
    denoise_wavelet,
    estimate_sigma,
    inpaint_biharmonic
)

# This will ignore some warnings caused by holoviews
warnings.simplefilter('ignore') 

sys.path.append('../../EMIT-Data-Resources/python/modules/')
from emit_tools import emit_xarray, ortho_xr

sys.path.append('../')
from config import *
from REFERENCE_PLANTS import REFERENCE_PLANTS

sys.path.append('../../datasets/')
import get_geosfp
import get_campd
import amf_compute
import importlib


# Reload the module
importlib.reload(get_campd)
importlib.reload(get_geosfp)
importlib.reload(amf_compute)

get_emission_rate = get_campd.get_emission_rate
get_emissions = get_campd.get_emissions

get_geosfp_wind = get_geosfp.get_geosfp_wind
get_geosfp_tph = get_geosfp.get_geosfp_tph

wav_min = 418
wav_max = 492
bin_size = 4
polydeg = 3

PREFIXES = {'OBS': 'EMIT_L1B_OBS_001_',
 'L1B': 'EMIT_L1B_RAD_001_',
 'MASK': 'EMIT_L2A_MASK_001_',
 'L2A': 'EMIT_L2A_RFL_001_'}

help(emit_xarray)

Help on function emit_xarray in module emit_tools:

emit_xarray(filepath, ortho=False, qmask=None, unpacked_bmask=None)
    This function utilizes other functions in this module to streamline opening an EMIT dataset as an xarray.Dataset.
    
    Parameters:
    filepath: a filepath to an EMIT netCDF file
    ortho: True or False, whether to orthorectify the dataset or leave in crosstrack/downtrack coordinates.
    qmask: a numpy array output from the quality_mask function used to mask pixels based on quality flags selected in that function. Any non-orthorectified array with the proper crosstrack and downtrack dimensions can also be used.
    unpacked_bmask: a numpy array from  the band_mask function that can be used to mask band-specific pixels that have been interpolated.
    
    Returns:
    out_xr: an xarray.Dataset constructed based on the parameters provided.



In [4]:
csv_savefn = f"{CONFIG['data_folder']}/metadata.csv"
data = pd.read_csv(csv_savefn)

dfs_merge = []

colormap_loc = {
    'Colstrip': '#1F77B4',                  # strong blue
    'Fort_Martin_Power_Station': '#FF7F0E', # vivid orange
    'Gerald_Gentleman_Station': '#2CA02C',  # strong green
    'Intermountain': '#D62728',             # strong red
    'Ghent': '#9467BD',                     # purple
    'Laramie_River': '#17BECF',             # cyan (distinct from blue)
    'Martin_Lake': '#7F7F7F',               # neutral gray
    'Mill_Creek': '#8C564B',                # brown
    'New_Madrid_Power_Plant': '#BCBD22',    # yellow-green/olive (bright)
    'Ninemile_Point': '#000000',            # black (max contrast)
    'Shawnee': '#E377C2',                   # pink/magenta
    'W_A_Parish': '#FFD700',                # gold (very bright, highly distinct)
}



for ln in colormap_loc.keys():
    dfl = data[data['LOC_NAME'] == ln]
    rlogfn = f'SourceRate_INIT/{ln}/retr_{ln}.csv'
    retr_log = pd.read_csv(rlogfn)

    dfs_merge.append(dfl.merge(
        retr_log,
        on="GRANULE",
        how="inner",
        validate="one_to_one"  # change to "many_to_one" if retr_log has unique GRANULE but df_loc doesn't
    ))

plot_df = pd.concat(dfs_merge, axis=0, ignore_index=True)
plot_df = plot_df[plot_df['AMF'] < 2.8]
# plot_df = plot_df[plot_df['AMF'] > 1]
plot_df

,LOC_NAME,GRANULE,LON,LAT,GOOD_DATA,PLUME,CAMPD_RATE,HRRR_AGL_DIR,HRRR_AGL_SPD,HRRR_10M_DIR,HRRR_10M_SPD,GEOSFP_50M_DIR,GEOSFP_50M_SPD,CAMPD,SMOOTHED,BINNED,MEAN,AMF
0,Colstrip,20250616T182554_2516712_006,-106.6140,45.8831,True,True,0.144594,4.221500,3.808876,1.786469,3.408175,358.442995,2.333290,0.144594,0.166101,0.155674,0.128496,1.226059
1,Colstrip,20250727T211648_2520814_022,-106.6140,45.8831,True,True,0.269627,74.373434,1.631814,60.646006,1.974344,9.597602,1.997187,0.269627,0.253020,0.237756,0.119715,1.089965
2,Colstrip,20240815T191038_2422813_023,-106.6140,45.8831,True,True,0.324455,318.962902,4.571384,323.278749,4.051774,314.212598,4.901754,0.324455,0.153678,0.154522,0.112532,1.569867
4,Fort_Martin_Power_Station,20240621T160640_2417311_031,-79.9275,39.7107,True,True,0.320582,255.581894,2.049553,241.913143,1.267794,271.447713,1.423885,0.320582,0.097123,0.096830,0.070768,1.469266
5,Fort_Martin_Power_Station,20250810T194210_2522213_031,-79.9275,39.7107,True,True,0.271314,153.450156,1.140011,158.274626,0.472135,230.940395,1.036547,0.271314,0.197885,0.194231,0.137723,0.701554
6,Gerald_Gentleman_Station,20230423T173615_2311312_010,-101.1408,41.0808,True,True,0.178451,280.925775,2.421152,289.590317,1.868374,278.054685,3.259158,0.178451,0.150995,0.172805,0.105072,1.515482
8,Gerald_Gentleman_Station,20250530T191434_2515013_020,-101.1408,41.0808,True,True,0.179758,205.321762,4.794936,198.717474,3.702924,227.516574,0.726616,0.179758,0.162895,0.169877,0.119516,1.250416
9,Gerald_Gentleman_Station,20250619T173827_2517011_024,-101.1408,41.0808,True,True,0.177455,200.126727,10.433670,197.196401,8.166441,200.263519,8.465850,0.177455,0.389229,0.439090,0.405393,1.159172
10,Intermountain,20231005T183601_2327812_013,-112.5811,39.5035,True,True,0.143605,25.604810,2.980168,354.100351,2.581978,16.761661,2.776960,0.143605,0.326888,0.313797,0.169773,1.571188
11,Intermountain,20240804T181803_2421712_014,-112.5811,39.5035,True,True,0.182110,224.778829,3.388981,221.150580,2.949279,252.962293,3.087215,0.182110,0.337370,0.334830,0.184838,1.568373


In [7]:
AMF_LUT = xr.load_dataset(CONFIG['AMF_LUT'])

def get_AMF(loc_name, granule_name):
    obs_time = datetime.strptime(granule_name.split('_')[0], "%Y%m%dT%H%M%S").replace(tzinfo=timezone.utc)
    loc_data = REFERENCE_PLANTS[loc_name]
    
    l2a_fpath  = f"{CONFIG['data_folder']}/{loc_name}/EMIT_L2A_RFL_001_{granule_name}.nc"
    L2A_xarr = emit_xarray(l2a_fpath, ortho=True)
    
    emit_spec_wlen = L2A_xarr["wavelengths"].to_numpy().astype(np.float64)
    window_sel = (emit_spec_wlen >= 405) & (emit_spec_wlen <= 465)
    gler = L2A_xarr['reflectance'].sel(wavelengths=window_sel, method='nearest').values
    gler = np.where(gler > -8000, gler, np.nan)
    gler = np.nanmean(gler, axis=-1)
    gler = np.nanmean(np.where(plume_mask_filled, gler, np.nan))
    
    del L2A_xarr
    
    obs_fpath  = f"{CONFIG['data_folder']}/{loc_name}/EMIT_L1B_OBS_001_{granule_name}.nc"
    OBS_xarr = emit_xarray(obs_fpath, ortho=True)
    obs_src = np.where(OBS_xarr['obs'] > -8000, OBS_xarr['obs'], np.nan)
    
    sza = np.nanmean(np.where(plume_mask_filled, obs_src[:,:,4], np.nan))
    vza = np.nanmean(np.where(plume_mask_filled, obs_src[:,:,2], np.nan))
    
    to_sensor_az = np.array(obs_src[:,:,1])
    to_sun_az = np.array(obs_src[:,:,3])
    rel_azimuth = np.abs(to_sensor_az-to_sun_az)
    rel_azimuth = np.minimum(rel_azimuth, 360.0 - rel_azimuth) # wrap to 0-180
    phi = np.nanmean(np.where(plume_mask_filled, rel_azimuth, np.nan))
    
    del OBS_xarr
    
    # Convert altitude to pressure via barometric formula with H=8km
    geosfp_info = get_geosfp_wind(loc_data['LAT'], loc_data['LON'], obs_time, cache=f'{CONFIG["geosfp"]}/')
    H = 287.05 * geosfp_info['T2M'] / 9.81
    surface_pressure = geosfp_info['PS']
    pressure_at_500m = surface_pressure * np.exp(-500/H)
    
    amf_profile = AMF_LUT["damf"].interp(dspres=surface_pressure, 
                                         dpres=pressure_at_500m,
                                         dalb=gler, dphi=phi, 
                                         dvza=vza, dsza=sza).data
    
    return amf_profile+0 # for some reason this typecasts it properly
    
    amf_eff

In [ ]:
plot_df['AMF_CORR'] = plot_df.apply(lambda row: get_AMF(row['LOC_NAME'], row['GRANULE']), axis=1)
plot_df.to_csv('ptsource_retr.csv', index=False)